# Generative Models — Simple Practical Notes
## Notebook 1: VAE, GAN and Diffusion

**Tools:** TensorFlow, Keras, NumPy and Matplotlib  
**Dataset:** MNIST handwritten digits  
**Recommended runtime:** Google Colab

### What we will build
1. A VAE that reconstructs and generates handwritten digits.
2. A GAN with a generator and discriminator.
3. A small demonstration of how diffusion adds noise to images.

The focus is on understanding the flow of data and the purpose of each line, rather than hiding the work inside many helper functions.

## 1. Import libraries

TensorFlow/Keras will build and train the neural networks. Matplotlib will display the images.

In [ ]:
# If needed, install TensorFlow:
# !pip install tensorflow matplotlib

import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

from tensorflow.keras import layers
from tensorflow.keras import Model

print("TensorFlow version:", tf.__version__)

## 2. Load MNIST data

Each image is 28 × 28 pixels and contains a handwritten digit.

We divide pixel values by 255 so they fall between 0 and 1.

In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

print("Training data:", x_train.shape)
print("Testing data:", x_test.shape)

In [ ]:
# Look at a few images before building the model.
plt.figure(figsize=(12, 3))

for i in range(10):
    plt.subplot(1, 10, i + 1)
    plt.imshow(x_train[i], cmap="gray")
    plt.axis("off")

plt.show()

# Part A — Variational Autoencoder (VAE)

## 3. Prepare the input

Our first VAE uses dense layers. A 28 × 28 image is therefore converted into a vector containing 784 values.

The same preprocessing is applied to the test images.

In [ ]:
# Flatten each 28 x 28 image into 784 values.
x_train_vae = x_train.reshape(-1, 784)
x_test_vae = x_test.reshape(-1, 784)

print(x_train_vae.shape)

## 4. Build the encoder

The encoder receives 784 values.

First, a Dense layer learns useful features. Then two separate Dense layers produce:
- `mean`: the centre of the latent distribution.
- `log_variance`: the spread of the latent distribution.

We use a latent dimension of 20, so each image is represented by two vectors of length 20.

In [ ]:
latent_dim = 20

# Input image: 784 values.
encoder_input = layers.Input(shape=(784,))

# Learn a hidden representation.
hidden = layers.Dense(400, activation="relu")(encoder_input)

# Produce the mean and log variance of the latent distribution.
mean = layers.Dense(latent_dim, name="mean")(hidden)
log_variance = layers.Dense(latent_dim, name="log_variance")(hidden)

# Create the encoder model.
encoder = Model(encoder_input, [mean, log_variance])

encoder.summary()

## 5. Sample a latent vector — reparameterization

We do not directly use the mean as the latent vector. We sample around the mean using the standard deviation and random noise.

\[
z = \mu + \sigma\epsilon
\]

The standard deviation is calculated from log variance:

\[
\sigma = \exp(0.5 \times \text{log variance})
\]

The random noise comes from a standard normal distribution.

In [ ]:
# Pass a sample image through the encoder.
sample_image = x_train_vae[:1]

mean_value, log_variance_value = encoder(sample_image)

# Convert log variance into standard deviation.
standard_deviation = tf.exp(0.5 * log_variance_value)

# Generate random noise with the same shape as the mean.
random_noise = tf.random.normal(shape=tf.shape(mean_value))

# Sample the latent vector.
z = mean_value + standard_deviation * random_noise

print("Mean shape:", mean_value.shape)
print("Log variance shape:", log_variance_value.shape)
print("Sampled latent vector shape:", z.shape)

## 6. Build the decoder

The decoder takes a 20-dimensional latent vector and converts it back into 784 values.

These 784 values represent the reconstructed image. We keep the final layer as logits; sigmoid will convert them into pixel probabilities when needed.

In [ ]:
decoder_input = layers.Input(shape=(latent_dim,))

decoder_hidden = layers.Dense(400, activation="relu")(decoder_input)

decoder_output = layers.Dense(784)(decoder_hidden)

decoder = Model(decoder_input, decoder_output)

decoder.summary()

## 7. Connect encoder and decoder

The complete flow is:

**Image → Encoder → Mean and log variance → Sample latent vector → Decoder → Reconstructed image**

The sampling operation is written directly in the model below.

In [ ]:
vae_input = layers.Input(shape=(784,))

# Encoder produces the two distribution parameters.
mean, log_variance = encoder(vae_input)

# Reparameterization: sample a latent vector.
standard_deviation = tf.exp(0.5 * log_variance)
random_noise = tf.random.normal(shape=tf.shape(mean))
z = mean + standard_deviation * random_noise

# Decoder reconstructs the image.
reconstruction = decoder(z)

# Complete VAE model.
vae = Model(vae_input, reconstruction)

vae.summary()

## 8. Understand the two parts of VAE loss

The VAE has two loss components.

**Reconstruction loss:** checks how close the reconstructed image is to the original image.

**KL divergence:** encourages the learned latent distribution to remain close to a standard normal distribution.

The total loss is the sum of these two terms.

We calculate the losses directly inside the training loop so it is easy to see how they are used.

## 9. Train the VAE

We use `GradientTape` because the loss contains both reconstruction loss and KL divergence.

Notice the sequence:
1. Pass a batch through the encoder and decoder.
2. Calculate reconstruction loss.
3. Calculate KL divergence.
4. Add the losses.
5. Calculate gradients.
6. Update the VAE weights.

In [ ]:
batch_size = 128
epochs = 10

train_dataset = (
    tf.data.Dataset.from_tensor_slices(x_train_vae)
    .shuffle(60000)
    .batch(batch_size)
)

optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

for epoch in range(epochs):

    total_loss = 0
    total_reconstruction_loss = 0
    total_kl_loss = 0
    batches = 0

    for batch in train_dataset:

        with tf.GradientTape() as tape:

            # Forward pass: input -> encoder -> latent sample -> decoder.
            mean, log_variance = encoder(batch)

            standard_deviation = tf.exp(0.5 * log_variance)
            random_noise = tf.random.normal(shape=tf.shape(mean))
            z = mean + standard_deviation * random_noise

            reconstruction = decoder(z)

            # Reconstruction loss: compare original pixels with reconstructed logits.
            reconstruction_loss = tf.reduce_sum(
                tf.nn.sigmoid_cross_entropy_with_logits(
                    labels=batch,
                    logits=reconstruction
                ),
                axis=1
            )
            reconstruction_loss = tf.reduce_mean(reconstruction_loss)

            # KL divergence: keep latent distributions close to N(0, I).
            kl_loss = -0.5 * tf.reduce_sum(
                1 + log_variance
                - tf.square(mean)
                - tf.exp(log_variance),
                axis=1
            )
            kl_loss = tf.reduce_mean(kl_loss)

            # Final VAE loss.
            loss = reconstruction_loss + kl_loss

        # Calculate gradients and update encoder + decoder weights.
        gradients = tape.gradient(loss, vae.trainable_variables)
        optimizer.apply_gradients(zip(gradients, vae.trainable_variables))

        total_loss += loss.numpy()
        total_reconstruction_loss += reconstruction_loss.numpy()
        total_kl_loss += kl_loss.numpy()
        batches += 1

    print(
        "Epoch:", epoch + 1,
        "| Total loss:", round(total_loss / batches, 2),
        "| Reconstruction:", round(total_reconstruction_loss / batches, 2),
        "| KL:", round(total_kl_loss / batches, 2)
    )

## 10. Reconstruct existing images

Now pass test images through the VAE.

The output is a reconstruction of each input image. Sigmoid converts the decoder's logits into values between 0 and 1.

In [ ]:
original_images = x_test_vae[:10]

reconstructed_images = vae.predict(original_images)
reconstructed_images = tf.sigmoid(reconstructed_images).numpy()

plt.figure(figsize=(14, 4))

for i in range(10):
    # Original image.
    plt.subplot(2, 10, i + 1)
    plt.imshow(original_images[i].reshape(28, 28), cmap="gray")
    plt.axis("off")

    # Reconstructed image.
    plt.subplot(2, 10, i + 11)
    plt.imshow(reconstructed_images[i].reshape(28, 28), cmap="gray")
    plt.axis("off")

plt.show()

## 11. Generate completely new images

For generation, we do not need an original image.

The latent space is encouraged to follow a standard normal distribution. We therefore generate random vectors from that distribution and pass them directly to the decoder.

In [ ]:
# Generate 10 random latent vectors.
random_latent_vectors = tf.random.normal(shape=(10, latent_dim))

# Send the random vectors directly to the decoder.
new_images = decoder.predict(random_latent_vectors)

# Convert logits into pixel values between 0 and 1.
new_images = tf.sigmoid(new_images).numpy()

plt.figure(figsize=(12, 3))

for i in range(10):
    plt.subplot(1, 10, i + 1)
    plt.imshow(new_images[i].reshape(28, 28), cmap="gray")
    plt.axis("off")

plt.show()

### VAE observations

- Compare original images with reconstructed images.
- Check whether generated images resemble handwritten digits.
- Change `latent_dim` to 2, 10, or 50 and train again.
- Change the number of epochs and compare the output.

**Remember:** the encoder is used during training and reconstruction. To generate a new image from a random latent vector, only the decoder is needed.

# Part B — Generative Adversarial Network (GAN)

## 12. Prepare the data for GAN

The generator will use `tanh` in its final layer. `tanh` produces values between -1 and 1.

Therefore, scale the MNIST images from their current range of 0 to 1 into the range -1 to 1.

In [ ]:
x_train_gan = (x_train_vae * 2.0) - 1.0

gan_dataset = (
    tf.data.Dataset.from_tensor_slices(x_train_gan)
    .shuffle(60000)
    .batch(128, drop_remainder=True)
)

noise_dimension = 100

## 13. Build the generator

The generator receives 100 random numbers.

It passes them through Dense layers and finally produces 784 values, which can be reshaped into a 28 × 28 image.

In [ ]:
generator = tf.keras.Sequential([
    layers.Input(shape=(noise_dimension,)),
    layers.Dense(256),
    layers.LeakyReLU(negative_slope=0.2),
    layers.BatchNormalization(),

    layers.Dense(512),
    layers.LeakyReLU(negative_slope=0.2),
    layers.BatchNormalization(),

    layers.Dense(1024),
    layers.LeakyReLU(negative_slope=0.2),
    layers.BatchNormalization(),

    layers.Dense(784, activation="tanh")
])

generator.summary()

## 14. Build the discriminator

The discriminator receives an image containing 784 values.

It predicts one output:
- A high probability/logit indicates real.
- A low probability/logit indicates generated.

The final layer has one neuron because this is a binary classification problem.

In [ ]:
discriminator = tf.keras.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(512),
    layers.LeakyReLU(negative_slope=0.2),
    layers.Dropout(0.3),

    layers.Dense(256),
    layers.LeakyReLU(negative_slope=0.2),
    layers.Dropout(0.3),

    layers.Dense(1)
])

discriminator.summary()

## 15. Understand the GAN training process

We train the two networks separately.

**First — train the discriminator**
1. Take real MNIST images and label them 1.
2. Generate fake images and label them 0.
3. Pass both through the discriminator.
4. Calculate the discriminator loss.
5. Update discriminator weights.

**Second — train the generator**
1. Generate a fresh batch of fake images.
2. Pass them through the discriminator.
3. Use label 1 because the generator wants the discriminator to classify fake images as real.
4. Calculate the generator loss.
5. Update generator weights.

During generator training, gradients pass through the discriminator to reach the generator. However, only generator weights are updated in that step.

## 16. Set up losses and optimizers

In [ ]:
binary_cross_entropy = tf.keras.losses.BinaryCrossentropy(from_logits=True)

generator_optimizer = tf.keras.optimizers.Adam(
    learning_rate=0.0002,
    beta_1=0.5
)

discriminator_optimizer = tf.keras.optimizers.Adam(
    learning_rate=0.0002,
    beta_1=0.5
)

## 17. Train the GAN

The two `GradientTape` blocks are deliberately written out.

In the first block, the tape watches the discriminator loss and we update discriminator weights.

In the second block, the tape watches the generator loss. The discriminator participates in the forward pass, but the optimizer updates only generator weights.

In [ ]:
epochs = 20

for epoch in range(epochs):

    discriminator_total_loss = 0
    generator_total_loss = 0
    batches = 0

    for real_images in gan_dataset:

        current_batch_size = real_images.shape[0]

        # -----------------------------------------
        # STEP 1: Train the discriminator
        # -----------------------------------------

        # Generate fake images from random noise.
        noise = tf.random.normal((current_batch_size, noise_dimension))
        fake_images = generator(noise, training=True)

        with tf.GradientTape() as discriminator_tape:

            # Discriminator predictions for real and fake images.
            real_predictions = discriminator(real_images, training=True)
            fake_predictions = discriminator(tf.stop_gradient(fake_images), training=True)

            # Real images should be labelled 1.
            real_loss = binary_cross_entropy(
                tf.ones_like(real_predictions),
                real_predictions
            )

            # Generated images should be labelled 0.
            fake_loss = binary_cross_entropy(
                tf.zeros_like(fake_predictions),
                fake_predictions
            )

            discriminator_loss = real_loss + fake_loss

        # Update only discriminator weights.
        discriminator_gradients = discriminator_tape.gradient(
            discriminator_loss,
            discriminator.trainable_variables
        )

        discriminator_optimizer.apply_gradients(
            zip(discriminator_gradients, discriminator.trainable_variables)
        )

        # -----------------------------------------
        # STEP 2: Train the generator
        # -----------------------------------------

        # Generate a fresh batch of random noise.
        noise = tf.random.normal((current_batch_size, noise_dimension))

        with tf.GradientTape() as generator_tape:

            generated_images = generator(noise, training=True)

            # Send generated images through the discriminator.
            predictions = discriminator(generated_images, training=True)

            # Generator wants generated images to be classified as real (label 1).
            generator_loss = binary_cross_entropy(
                tf.ones_like(predictions),
                predictions
            )

        # Update only generator weights.
        generator_gradients = generator_tape.gradient(
            generator_loss,
            generator.trainable_variables
        )

        generator_optimizer.apply_gradients(
            zip(generator_gradients, generator.trainable_variables)
        )

        discriminator_total_loss += discriminator_loss.numpy()
        generator_total_loss += generator_loss.numpy()
        batches += 1

    print(
        "Epoch:", epoch + 1,
        "| Discriminator loss:", round(discriminator_total_loss / batches, 4),
        "| Generator loss:", round(generator_total_loss / batches, 4)
    )

## 18. Generate images using the trained GAN

The generator does not need a real image. It only needs random noise.

The generated output is in the range -1 to 1, so convert it back to 0 to 1 before displaying.

In [ ]:
noise = tf.random.normal((10, noise_dimension))

generated_images = generator.predict(noise)

# Convert from [-1, 1] to [0, 1].
generated_images = (generated_images + 1.0) / 2.0

plt.figure(figsize=(12, 3))

for i in range(10):
    plt.subplot(1, 10, i + 1)
    plt.imshow(generated_images[i].reshape(28, 28), cmap="gray")
    plt.axis("off")

plt.show()

### GAN observations

- Check whether the generated digits become more recognizable over time.
- Notice whether the generator produces varied digits or repeatedly produces similar images.
- Compare generator and discriminator losses, but do not judge image quality using losses alone.

**Weight update reminder**
- Discriminator training updates discriminator weights.
- Generator training updates generator weights.
- During generator training, the discriminator still passes gradients back to the generator.

# Part C — Diffusion Fundamentals

## 19. Understand forward diffusion

Diffusion gradually adds random noise to an image.

At the beginning, the image is clear. As the timestep increases, more noise is added. Eventually, the image becomes difficult to recognize.

For this first practical exercise, we will visualize the forward process directly. This keeps the main idea clear before moving to pretrained diffusion models.

In [ ]:
# Select one image and display the original.
image = x_train[0]

plt.imshow(image, cmap="gray")
plt.title("Original image")
plt.axis("off")
plt.show()

## 20. Add noise gradually

At each step, we add a little Gaussian noise to the image.

This is a simplified visualization of the forward diffusion process. A full diffusion implementation uses a defined noise schedule and a corresponding reverse denoising process.

In [ ]:
# Start with the original image.
noisy_image = image.copy()

# Display the image after different amounts of noise.
plt.figure(figsize=(14, 3))

for i, noise_level in enumerate([0.1, 0.2, 0.3, 0.5, 0.8]):

    # Generate random noise with the same shape as the image.
    random_noise = np.random.normal(
        loc=0.0,
        scale=1.0,
        size=image.shape
    )

    # Mix the original image with random noise.
    noisy_image = (1 - noise_level) * image + noise_level * random_noise

    plt.subplot(1, 5, i + 1)
    plt.imshow(noisy_image, cmap="gray")
    plt.title("Noise: " + str(noise_level))
    plt.axis("off")

plt.show()

## 21. Compare the three models

| Model | What it learns | How new images are generated |
|---|---|---|
| VAE | A structured latent distribution and a decoder | Sample a latent vector and pass it to the decoder |
| GAN | A generator that can fool a discriminator | Pass random noise to the generator |
| Diffusion | How noise relates to images at different noise levels | Start with noise and repeatedly denoise it |

### Quick review
1. Why does the VAE encoder produce mean and log variance?
2. Why do we need the reparameterization trick?
3. Which weights are updated during discriminator training?
4. Why does the generator use label 1 for generated images during its training step?
5. What happens to an image during forward diffusion?
6. How does generation differ between VAE and GAN?

### Practical outputs
- VAE reconstruction comparison.
- VAE-generated digits.
- GAN-generated digits.
- Diffusion noise progression.